# Water Quality Indices from Sentinel-2 L2A via openEO

Retrieves Sentinel-2 L2A imagery over a user-defined area and time window using the
[Copernicus Data Space Ecosystem (CDSE) openEO backend](https://openeo.dataspace.copernicus.eu),
applies cloud masking, and computes three water quality indices:

| Index | Full name | Target parameter |
|-------|-----------|-----------------|
| **NDWI** | Normalized Difference Water Index | Water body mask |
| **NDCI** | Normalized Difference Chlorophyll Index | Chlorophyll-*a* proxy |
| **Three-band model** | Dall'Olmo / Gitelson | Chlorophyll-*a* proxy |
| **FAI** | Floating Algae Index | Surface algae / cyanobacteria |

**Requirements:** `openeo`, `rioxarray`, `xarray`, `matplotlib`, `pandas`

```
pip install openeo rioxarray xarray matplotlib pandas
```


## 1  Connect and authenticate

Authentication uses OpenID Connect (OIDC). On first run a browser window will open; subsequent runs reuse the cached token.

In [ ]:
import openeo

connection = openeo.connect("openeo.dataspace.copernicus.eu")

try:
    connection = connection.authenticate_oidc(
        max_poll_time=60,
        display=True
    )
    print("✅ Authentication successful")
except Exception as e:
    print(f"❌ Authentication failed: {e}")


## 2  (Optional) Explore available Sentinel collections

In [ ]:
# List all Sentinel collections available on the backend
sentinel_collections = [c for c in connection.list_collection_ids() if "SENTINEL" in c]
print(sentinel_collections)


In [ ]:
# Inspect the Sentinel-2 L2A collection metadata
connection.describe_collection("SENTINEL2_L2A")


## 3  User parameters

Edit the bounding box and date range here — everything else runs automatically.

In [ ]:
# ── Spatial extent ──────────────────────────────────────────────────────────
# Default: Lake Garda, Italy  (lon 10.5–10.9, lat 45.4–45.9)
lon_min, lon_max = 10.5, 10.9
lat_min, lat_max = 45.4, 45.9

spatial_extent = {
    "west":  lon_min,
    "east":  lon_max,
    "south": lat_min,
    "north": lat_max,
}

# ── Temporal extent ──────────────────────────────────────────────────────────
temporal_extent = ["2025-08-01", "2025-08-30"]   # [start, end]  "YYYY-MM-DD"

# ── Cloud cover threshold ────────────────────────────────────────────────────
MAX_CLOUD_COVER = 10  # percent


## 4  Load Sentinel-2 L2A data cube

In [ ]:
sentinel2_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=[
        "B02",  # Blue         ~490 nm
        "B03",  # Green        ~560 nm
        "B04",  # Red          ~665 nm
        "B05",  # Red Edge 1   ~705 nm
        "B06",  # Red Edge 2   ~740 nm
        "B07",  # Red Edge 3   ~783 nm
        "B08",  # NIR          ~842 nm
        "B8A",  # Narrow NIR   ~865 nm
        "B11",  # SWIR 1       ~1610 nm
        "B12",  # SWIR 2       ~2190 nm
        "SCL",  # Scene Classification Layer
    ],
    max_cloud_cover=MAX_CLOUD_COVER,
)


## 5  Cloud masking and reflectance scaling

`mask_scl_dilation` removes cloud, cloud shadow, and snow pixels using the SCL band with a small dilation buffer.  
DN values are then converted to surface reflectance by multiplying by 0.0001.

In [ ]:
from openeo.processes import ProcessBuilder

# --- Cloud / shadow masking via SCL dilation ---
sentinel2_cube = sentinel2_cube.process(
    "mask_scl_dilation",
    data=sentinel2_cube,
    scl_band_name="SCL"
)

# --- DN → surface reflectance (factor 0.0001) ---
def scale_to_reflectance(x: ProcessBuilder):
    return x * 0.0001

reflectance_cube = sentinel2_cube.apply(scale_to_reflectance)


## 6  Create and submit the processing job

In [ ]:
final_results = reflectance_cube.save_result(format="GTiff")

job = final_results.create_job(
    title="WaterQuality_S2L2A",
    description="Sentinel-2 L2A reflectance stack for water quality index computation"
)

# Display the process graph
job


In [ ]:
try:
    result = job.start_and_wait(
        print=lambda msg: print(f"[{msg}]"),
        max_poll_interval=30,
        connection_retry_interval=60,
    )
    print(f"✅ Job completed — ID: {job.job_id}")
except Exception as e:
    print(f"❌ Job failed: {e}")


## 7  Download results

In [ ]:
# Set your local download directory
download_dir = "./data/sentinel2_results"   # change as needed

results = job.download_results(download_dir)
print(f"Downloaded {len(results)} file(s) to '{download_dir}/'")


## 8  Load downloaded GeoTIFFs into an xarray stack

Each file name encodes the acquisition date (format: `*_<YYYYMMDDTHHMMSS>Z_*.tif`).  
Files are sorted chronologically and stacked along a `time` dimension.

In [ ]:
import pandas as pd
import rioxarray as rxr
import xarray as xr

images, times = [], []

for result in sorted(results):
    images.append(rxr.open_rasterio(result))
    date_str = str(result).split("_")[1].split("Z")[0]   # extract datetime token
    times.append(pd.to_datetime(date_str))

stacked = xr.concat(images, dim="time")
stacked = stacked.assign_coords(time=times)

print("Dimensions:", stacked.dims)
print("Shape:     ", stacked.shape)
print("Band order:", stacked.band.values)


## 9  Quick RGB preview (first date)

In [ ]:
import matplotlib.pyplot as plt

# Band indices: 1=B02, 2=B03, 3=B04 (Blue, Green, Red → displayed as RGB)
stacked[0].sel(band=[3, 2, 1]).plot.imshow(robust=True)
plt.title(f"RGB composite — {str(stacked.time.values[0])[:10]}")
plt.axis("off")
plt.tight_layout()
plt.show()


## 10  NDWI water mask

$$NDWI = \frac{R_{rs}(560) - R_{rs}(842)}{R_{rs}(560) + R_{rs}(842)}$$

Pixels with NDWI ≤ 0.1 are masked out, retaining open water only.

In [ ]:
# Band 2 = Green (B03, ~560 nm),  Band 7 = NIR (B08, ~842 nm)
NDWI = (stacked.sel(band=2) - stacked.sel(band=7)) / (
        stacked.sel(band=2) + stacked.sel(band=7))

stacked_masked = stacked.where(NDWI > 0.1)

n_dates = len(stacked_masked)
fig, axes = plt.subplots(1, n_dates, figsize=(5 * n_dates, 5))
if n_dates == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    stacked_masked[i].sel(band=[3, 2, 1]).plot.imshow(ax=ax, robust=True)
    ax.set_title(f"RGB + NDWI mask\n{str(stacked.time.values[i])[:10]}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 11  NDCI — Normalized Difference Chlorophyll Index

$$NDCI = \frac{R_{rs}(709) - R_{rs}(665)}{R_{rs}(709) + R_{rs}(665)}$$

Uses Red Edge 1 (B05, ~705 nm) and Red (B04, ~665 nm).  
Higher values indicate elevated chlorophyll-*a* concentrations.

In [ ]:
# Band 4 = B05 (Red Edge 1, ~705 nm),  Band 3 = B04 (Red, ~665 nm)
ndci = (stacked_masked.sel(band=4) - stacked_masked.sel(band=3)) / (
        stacked_masked.sel(band=4) + stacked_masked.sel(band=3))

vmin, vmax = float(ndci.min()), float(ndci.max())

fig, axes = plt.subplots(1, n_dates, figsize=(5 * n_dates, 5))
if n_dates == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ndci[i].plot.imshow(ax=ax, vmin=vmin, vmax=vmax, cmap="RdYlGn")
    ax.set_title(f"NDCI\n{str(stacked.time.values[i])[:10]}")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Spatial median time series
ndci_median = ndci.median(dim=["x", "y"])

ndci_median.plot(marker="o", color="red", linestyle="-.")
plt.title("NDCI — spatial median time series")
plt.xlabel("Date")
plt.ylabel("NDCI")
plt.grid(True)
plt.tight_layout()
plt.show()


## 12  Three-band model — Chlorophyll-*a* (Dall'Olmo / Gitelson)

$$Chl_a \propto \left(\frac{1}{R_{rs}(670)} - \frac{1}{R_{rs}(710)}\right) \cdot R_{rs}(740)$$

Uses B04 (~665 nm), B05 (~705 nm), and B06 (~740 nm).  
Output is a proxy — apply an empirical regression to convert to mg m⁻³.

In [ ]:
# Band 3 = B04 (~665 nm),  Band 4 = B05 (~705 nm),  Band 5 = B06 (~740 nm)
chl = ((1 / stacked_masked.sel(band=3)) - (1 / stacked_masked.sel(band=4))) *        stacked_masked.sel(band=5)

vmin, vmax = float(chl.min()), float(chl.max())

fig, axes = plt.subplots(1, n_dates, figsize=(5 * n_dates, 5))
if n_dates == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    chl[i].plot.imshow(ax=ax, vmin=vmin, vmax=vmax, cmap="YlGn")
    ax.set_title(f"3-band Chl-a model\n{str(stacked.time.values[i])[:10]}")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
chl_median = chl.median(dim=["x", "y"])

chl_median.plot(marker="o", color="green", linestyle="-.")
plt.title("Three-band Chl-a model — spatial median time series")
plt.xlabel("Date")
plt.ylabel("Three-band index (proxy)")
plt.grid(True)
plt.tight_layout()
plt.show()


## 13  FAI — Floating Algae Index

$$FAI = R_{rs}(842) - \left[ R_{rs}(665) + \left(R_{rs}(1610) - R_{rs}(665)\right) \cdot \frac{842 - 665}{1610 - 665} \right]$$

Uses NIR (B08, ~842 nm), Red (B04, ~665 nm), and SWIR-1 (B11, ~1610 nm).  
Positive FAI values indicate floating algae or cyanobacterial mats.

In [ ]:
# Band 3 = B04 (Red, ~665 nm),  Band 7 = B08 (NIR, ~842 nm),  Band 9 = B11 (SWIR-1, ~1610 nm)
red  = stacked_masked.sel(band=3)
nir  = stacked_masked.sel(band=7)
swir = stacked_masked.sel(band=9)

fai = nir - (red + (swir - red) * ((842 - 665) / (1610 - 665)))

vmin, vmax = float(fai.min()), float(fai.max())

fig, axes = plt.subplots(1, n_dates, figsize=(5 * n_dates, 5))
if n_dates == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    fai[i].plot.imshow(ax=ax, vmin=vmin, vmax=vmax, cmap="BuPu")
    ax.set_title(f"FAI\n{str(stacked.time.values[i])[:10]}")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fai_median = fai.median(dim=["x", "y"])

fai_median.plot(marker="o", color="blue", linestyle="-.")
plt.title("FAI — spatial median time series")
plt.xlabel("Date")
plt.ylabel("FAI")
plt.grid(True)
plt.tight_layout()
plt.show()
